In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from astropy import units as u
import itertools
import pandas as pd

In [2]:
stem = '/Users/eckhartspalding/Documents/git.repos/nice/data/'

In [3]:
# filter combos:

'''
# any of the filters by themselves:
1
2
0.3
3

# either of the DataRay filters, plus one or both of the Thorlabs filters:
1+0.3
2+0.3
1+3
2+3
1+0.3+3
2+0.3+3

# Thorlabs filters together
0.3+3
'''

'\n# any of the filters by themselves:\n1\n2\n0.3\n3\n\n# either of the DataRay filters, plus one or both of the Thorlabs filters:\n1+0.3\n2+0.3\n1+3\n2+3\n1+0.3+3\n2+0.3+3\n\n# Thorlabs filters together\n0.3+3\n'

In [4]:
# net combinations of ND:
nds_possible = {'NE0.3': 0.3, 'NE1': 1, 'NE2': 2, 'NE3': 3, 
    'NE1.3': 1.3, 'NE2.3': 2.3, 'NE3.3': 3.3, 'NE4': 4, 'NE5': 5, 'NE4.3': 4.3, 'NE5.3': 5.3}
nds_possible = pd.DataFrame(list(nds_possible.items()), columns=['Filters', 'Total_ND'])

In [5]:
# Ref. doc https://assets.dataray.com/pdf/dataray-wincamd-ir-bb-app-note.pdf
# - for transmission curves, see Fig. 4
# - for detector response, see Fig. 6

In [6]:
target_um = 3.9 # wavelength of interest

In [7]:
# include effect of power levels

power_levels = np.array([1.5, 7.5, 15.0, 75.0, 150.0, 183.0]) * u.mW # set by laser

# Create a list to store the expanded rows
expanded_rows = []

# Enumerate every row in combos_nds_df with each power level
for idx, row in nds_possible.iterrows():
    for power in power_levels:
        expanded_rows.append({
            'Filters': row['Filters'],
            'Total_ND': row['Total_ND'],
            'Power_mW_emitted': power.value  # store as float for easy analysis
        })

# Create the expanded dataframe
combos_power_df = pd.DataFrame(expanded_rows)

# add column of received power at the detector
combos_power_df['Power_mW_received'] = combos_power_df['Power_mW_emitted'] * 10**(-combos_power_df['Total_ND'])


combos_power_df.sort_values(by='Power_mW_received', ascending=True, inplace=True)

In [8]:
pd.set_option('display.max_rows', 500)
print(combos_power_df[['Filters', 'Power_mW_emitted', 'Power_mW_received']])

   Filters  Power_mW_emitted  Power_mW_received
60   NE5.3               1.5           0.000008
48     NE5               1.5           0.000015
61   NE5.3               7.5           0.000038
49     NE5               7.5           0.000075
62   NE5.3              15.0           0.000075
54   NE4.3               1.5           0.000075
50     NE5              15.0           0.000150
42     NE4               1.5           0.000150
63   NE5.3              75.0           0.000376
55   NE4.3               7.5           0.000376
43     NE4               7.5           0.000750
51     NE5              75.0           0.000750
36   NE3.3               1.5           0.000752
56   NE4.3              15.0           0.000752
64   NE5.3             150.0           0.000752
65   NE5.3             183.0           0.000917
44     NE4              15.0           0.001500
18     NE3               1.5           0.001500
52     NE5             150.0           0.001500
53     NE5             183.0           0

In [37]:
# Note 185 uW led to ~50% max ADU on the detector; so the incident power could be as high as ~260 uW (~70% of max well)

power_lower_limit = 7.5e-3 * u.mW
power_upper_safe_limit = 1. * u.mW
combos_power_df_useable_range = combos_power_df[
    (combos_power_df['Power_mW_received'] > (power_lower_limit)) &
    (combos_power_df['Power_mW_received'] < (power_upper_safe_limit
))
]

In [38]:
print(combos_power_df_useable_range)

   Filters  Total_ND  Power_mW_emitted  Power_mW_received
45     NE4       4.0              75.0           0.007500
30   NE2.3       2.3               1.5           0.007518
58   NE4.3       4.3             150.0           0.007518
38   NE3.3       3.3              15.0           0.007518
59   NE4.3       4.3             183.0           0.009172
12     NE2       2.0               1.5           0.015000
20     NE3       3.0              15.0           0.015000
46     NE4       4.0             150.0           0.015000
47     NE4       4.0             183.0           0.018300
31   NE2.3       2.3               7.5           0.037589
39   NE3.3       3.3              75.0           0.037589
21     NE3       3.0              75.0           0.075000
13     NE2       2.0               7.5           0.075000
24   NE1.3       1.3               1.5           0.075178
40   NE3.3       3.3             150.0           0.075178
32   NE2.3       2.3              15.0           0.075178
41   NE3.3    